# components.collection

> Main entry point for rendering a virtual collection.

In [ ]:
#| default_exp components.collection

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Optional

from fasthtml.common import Div, Hidden

from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_direction, flex
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.components.table import render_header_row, render_table_rows
from cjm_fasthtml_virtual_collection.components.footer import render_footer
from cjm_fasthtml_virtual_collection.components.scrollbar import render_scrollbar

In [ ]:
#| export
def render_virtual_collection(
    items: list,                                # Full item list
    config: VirtualCollectionConfig,             # Collection config
    state: VirtualCollectionState,               # Collection state
    ids: VirtualCollectionHtmlIds,               # HTML IDs
    urls: VirtualCollectionUrls,                 # URL bundle
    render_cell: Optional[Callable] = None,      # Table layout cell render callback
    render_item: Optional[Callable] = None,      # Grid layout item render callback
) -> Div:  # Complete collection element
    """Render a complete virtual collection with header, viewport, scrollbar, and footer."""
    children = []

    if config.layout == "table":
        assert render_cell is not None, "render_cell required for table layout"

        # Header (fixed, outside viewport)
        children.append(render_header_row(config, ids))

        # Viewport body (rows) + scrollbar side-by-side
        rows = render_table_rows(items, config, state, ids, render_cell, focus_url=urls.focus_row)
        viewport = Div(
            rows,
            id=ids.viewport,
            cls=combine_classes(flex(1), overflow.hidden),
        )

        if config.show_scrollbar and state.total_items > state.visible_rows:
            scrollbar = render_scrollbar(state, config, ids)
            body = Div(viewport, scrollbar, cls=combine_classes(flex_display))
        else:
            body = viewport

        children.append(body)

    elif config.layout == "grid":
        raise NotImplementedError("Grid layout will be added in Phase 14")
    else:
        raise ValueError(f"Unknown layout: {config.layout}")

    # Footer
    children.append(render_footer(state, ids))

    # Hidden input for JS thumb positioning (OOB-updated on navigation)
    children.append(Hidden(
        value=str(state.window_start),
        id=ids.window_start_input,
        name="window_start",
    ))

    return Div(
        *children,
        id=ids.collection,
        cls=combine_classes(flex_display, flex_direction.col),
    )

## Tests

In [ ]:
from dataclasses import dataclass
from fasthtml.common import to_xml, Span
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

@dataclass
class Item:
    name: str
    size: str

test_items = [Item(f"file_{i}.txt", f"{i}KB") for i in range(20)]

def test_render_cell(item, ctx):
    if ctx.column.key == "name": return Span(item.name)
    if ctx.column.key == "size": return Span(item.size)

columns = (
    ColumnDef(key="name", header="Name", width="1fr"),
    ColumnDef(key="size", header="Size", width="100px"),
)
config = VirtualCollectionConfig(prefix="tc", columns=columns, row_height=40)
state = VirtualCollectionState(total_items=20, visible_rows=5)
ids = VirtualCollectionHtmlIds(prefix=config.prefix)
urls = VirtualCollectionUrls()

In [ ]:
# Test full collection render
collection = render_virtual_collection(
    items=test_items, config=config, state=state,
    ids=ids, urls=urls, render_cell=test_render_cell,
)
html = to_xml(collection)

# Outer container
assert 'id="tc-collection"' in html

# Header
assert 'id="tc-header"' in html
assert 'Name' in html
assert 'Size' in html

# Viewport with rows
assert 'id="tc-viewport"' in html
assert 'id="tc-rows"' in html

# Visible rows (0-4, visible_rows=5)
assert 'id="tc-row-0"' in html
assert 'id="tc-row-4"' in html
assert 'id="tc-row-5"' not in html  # Not visible

# Cell IDs
assert 'id="tc-row-0-col-name"' in html
assert 'id="tc-row-0-col-size"' in html

# Scrollbar (20 items > 5 visible, so scrollbar should be shown)
assert 'id="tc-scrollbar-track"' in html
assert 'id="tc-scrollbar-thumb"' in html

# Footer
assert 'id="tc-footer"' in html
assert 'Showing 1-5 of 20 items' in html

# Cell content
assert 'file_0.txt' in html
assert 'file_4.txt' in html
assert 'file_5.txt' not in html

print("Full collection render test passed")

In [ ]:
# Test with window_start offset
state2 = VirtualCollectionState(total_items=20, visible_rows=5, window_start=10)
collection2 = render_virtual_collection(
    items=test_items, config=config, state=state2,
    ids=ids, urls=urls, render_cell=test_render_cell,
)
html2 = to_xml(collection2)

assert 'id="tc-row-10"' in html2
assert 'id="tc-row-14"' in html2
assert 'id="tc-row-9"' not in html2
assert 'id="tc-row-15"' not in html2
assert 'Showing 11-15 of 20 items' in html2
assert 'file_10.txt' in html2

print("Offset window render test passed")

Offset window render test passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()